# Résiliation client SaaS (churn)

## 1. Informations

Certification : Aelion - Concervoir et implémenter une solution d'IA.  
Candidat : Aurélien Vallet  
Date de remise : 02/10/2026  
Version du notebook : 1

## 2. Présentation du sujet

### 2.1 Présentation globale

Le projet "churn" à pour contexte un éditeur de logiciel SaaS qui commercialise ses solutions via un système d'abonnement. La résilisation des abonnements par les client est donc la problématique principale de ce sujet car elle a un impacte directement sur la rentabilité de l'éditeur.

Un extrait de la base de donnée client nous est fournit au format csv (le fichier est présent dans `/docs/churn_saas_complet.csv` et `catalogue_plans.csv`) contenant respectivement l'ensemble des clients et les données des abonnements.

### 2.2 Besoin métier

En tant que développeur de solution Machine Learning capable, mon rôle est de répondre aux besoins du client en lui proposant une solution capable de lui faire obtenir de la plus values sur son business, ces outils sont à destinations des Customer Success Manager (CSM).

Ma solution Machine Learning devra permettre au CSM de :
- détecter les clients susceptibles de résilier leur abonnements à un instant T
- estimer ce qu'un client devrait lui rapporter au cours de sa vie client

Les métriques suivantes déterminerons si la solutions IA est efficaces :
- Taux de churn : clients ayant résiliés / clients total, nous utiliserons le taux de churn actuel (28%) comme celui de référence, l'objectif est de rester sous ce chiffre.
- Nombre de client sauvés : nombre de client qui souhaités résilier leur abonnement mais ont changés d'avis
- Ecart estimation CLV - CLV a un instant , le client est en mesure de savoir si le potentiel commercial d'un client est atteint ou non.

###  2.3 FN VS FP

Faux positifs = le modèle prédit une resiliation alors que le client ne prévoit pas de résilier (fausse alerte)
Faux négatifs = le modèle prédit une non-résiliation alors que le client prévoit de résilier (résiliation manquée)

Les faux positifs sont moins grave que les faux négatifs car ils impliquent principalement du temps d'investigation de la part du CSM et un dérangement du client, alors que les faux négatifs impliquent une perte de chiffre d'affaire immédiate.

### 2.4 Production IA

L'apprentissage supervisé sera priviligié dans notre cas car la donnée de "churn" est présente dans les données client.
Deux modèles de Machine Learning seront produit et livrés à l'équipe CSM :
- Modèle de prédiction du churn client
- Modèle d'estimation de CLV

Ces deux modèles seront intégrés dans un interface graphique afin de permettre aux CSM d'intérroger les modèles.

### 2.5 Hypothèses

La resiliation d'un client est selon moi causée par l'ensemble de ces critères :
- satisfaction client
    - support de qualité (peu de demande et traitement rapide) : `tickets_support_90j`,`delai_reponse_support_h` 
    - les services offert sont utilisés et il y a une part importante d'utilisateur actif (correspond à la demande de l'utilisateur) : `fonctionnalites_utilisees`,`taux_adoption_pct` 
    - appreciation du produit : `csat`
- Facteur économique
    - grosse entreprise : les grosses entreprises ont de moyens qui leurs permettent d'acheter des licences et ne pas se soucier du retour sur investissement `taille_entreprise` 
    - le retard de paiement indique que le client à des problèmes pour payer et il envisage surement la résiliation pour alléger ses dépenses : `retards_paiement_12m` 

### 2.6 Enjeux éthique

Impact écologique : les temps de traitement et les coûts écologiques (CO2 émis et électricité consommé) liés à l'entrainement des modèles sont presque négligeable dans notre cas étant donnée la rapidité avec laquelle ceux-ci sont réalisés.

Risque d'érreur : les modèles développés ne sont pas des solutions 100% fiable, il à une marge d'érreur qui peut induire des résultat faux, pour estimer ce pourcentage il faut se rapporter aux différents indicateurs présents dans chaque modèle (Recall, FP, FN).

Données sensibles : la solution IA ne traite aucune données à caractère personnelles, le RGPD ne n'applique pas.

### 2.7 Difficultés

Ce sujet présente certaines difficultés :
- Aucun contact avec le client, le sujet est statique il faut donc supputer certains données et prendre certaines décisions à la place du client.

## 3. 🥉🥈🥇 Ingestion complète

Lance les trois couches à la suite : `docs/*.csv` → 🥉 bronze → 🥈 silver →
🥇 gold. Chaque couche est intégralement rechargée, l'exécution est donc
rejouable sans créer de doublons.

In [ ]:
from ml_churn.ingestion.scripts.ingest_all import ingest_all

lignes_ingestion = ingest_all()

## 4. Exploration des données brutes

Deux types de graphiques construits sur la couche bronze, où les données brutes
sont stockées telles quelles : l'ingestion bronze doit donc avoir tourné. Les
valeurs sont normalisées à la volée — les données mélangent les casses
(`TPE`, `tpe`, `" TPE "`) et les unités (`20.0%`, `3.1 h`, `280.62 €`).

On distingue deux types de graphiques :

**Histogrammes** — ils nous permettent de voir le type de distribution d'un type de donnée, répartition des clients par secteur, pays, taille d'entreprise et plan, puis distribution des variables d'usage (ancienneté,
CSAT, délai de réponse du support, revenu mensuel…).

**Boîtes à moustaches** — ce graphique nous permet de visualiser les outliers dans une distribution, le boxplot isole individuellement chaque point au-delà de 1,5 × IQR.

Cette première analyse exploration des données nous sera utile lors de l'imputation de la couche silver pour déterminer certains choix de criètre d'imputation (moyenne, médiane...).

Les PNG sont exportés dans `src/visualization/graphs/<type>/<préfixe>_<donnée>.png`.

In [ ]:
import sys

sys.path.insert(0, "src")  # src/visualization n'est pas un package installe
from visualization.scripts.plot_all import plot_all

graphiques = plot_all()

## 5. 🥉 Ingestion — couche bronze

Les trois CSV de `docs/` sont copiés **tels quels** dans le schéma `bronze` :
toutes les colonnes métier en `TEXT`, aucune conversion ni nettoyage. Chaque ligne
conserve son origine (`_source_file`, `_source_line`, `_ingested_at`).

Les tables sont vidées puis rechargées : relancer la cellule ne crée pas de
doublons. Le log compare les lignes lues dans le CSV à celles réellement insérées
en base, et signale tout écart.

In [ ]:
from ml_churn.ingestion.scripts.ingest_bronze import ingest_bronze

lignes_bronze = ingest_bronze()

## 6. 🥈 Ingestion — couche silver

`bronze.churn_saas_complet_bronze` → `silver.churn_saas_silver`, en onze actions
successives :

1. **Déduplication** sur `client_id`
2. **Standardisation** de huit colonnes : dates ramenées au format `AAAA-MM-JJ`,
   pays en codes ISO, plans en `PRO`/`BUS`/`STR`/`ENT`, casse et espaces harmonisés
3. **Règles métier** : toute ligne hors bornes est supprimée
4. **Typage** : `date`, `integer` et `numeric` selon la colonne

In [ ]:
from ml_churn.ingestion.scripts.ingest_silver import ingest_silver

lignes_silver = ingest_silver()

## 7. 🥇 Ingestion — couche gold

`silver` → `gold`, pour les deux tables : `catalogue_gold` et
`churn_saas_gold`. Les tables reprennent **la structure de silver à
l'identique** (mêmes colonnes, mêmes types) — les définitions sont partagées
via les mixins de `models/colonnes.py`, il n'y a donc pas deux schémas à
maintenir.

Gold est la couche stable sur laquelle s'appuient l'analyse et la
modélisation, sans dépendre des retraitements successifs de silver.

In [ ]:
from ml_churn.ingestion.scripts.ingest_gold import ingest_gold

lignes_gold = ingest_gold()

## 8. Déséquilibre de la cible

Effectifs de chaque valeur de `churn` et taux de résiliation, lus dans
`gold.churn_saas_gold`. Le code est dans `src/ml_churn/training/`.

Ce déséquilibre conditionne le choix des métriques : un modèle qui prédirait
« personne ne résilie » aurait déjà une exactitude égale à la part de clients
actifs.

In [ ]:
from ml_churn.training.classification.analyze_target import analyze_target

repartition_cible = analyze_target()

## 9. Choix des Modeles

Pour le problème de classification voici la liste des modèles dont j'ai à ma disposition :

- regression logistique
- Random Forest
- XG Boost
- Gradient boosting

Pour le modèle baseline de référence je choisis de prendre la régréssion logistique car il s'agit d'un modèle simple à mettre en oeuvre et permet de résoudre un prblème catégoriel.
Le second modèle sera XGBoost car il gère les déséquilibre de classe et les meilleures performances en termce de prédiction.


## 10. Modèle de Classification

### 10.1 Entrainement du modèle baseline (régression logistique)

**Entraînement du modèle baseline.** Un modèle de regréssion logistique est utilisé pour avoir un point de référence de comparaison pour les modèles plus complèxes.
Tous les paramètres du modèle sont laissés par défaut (seuil a 0.5) et le modèle est enregistré dans `artifacts/baseline/<date>/`.

In [ ]:
from ml_churn.training.classification.baseline.classification_baseline_training import (
    train_classification_baseline,
)
from ml_churn.training.common.explain import (
    display_figure,
    shap_bar_figure,
    shap_summary_figure,
)

# Seuil par defaut : 0.5.
resultat_baseline = train_classification_baseline()

# Explication sur le jeu de test : ce que le modele a appris se juge sur des
# lignes qu'il n'a pas vues.
display_figure(
    shap_bar_figure(
        resultat_baseline.model,
        resultat_baseline.X_test,
        titre=f"SHAP — importance globale (seuil {resultat_baseline.seuil:.2f})",
        max_features=None,  # toutes les colonnes, sans regroupement
    )
)
display_figure(
    shap_summary_figure(
        resultat_baseline.model,
        resultat_baseline.X_test,
        titre=f"SHAP — effet par client (seuil {resultat_baseline.seuil:.2f})",
    )
)


**Explication des features du baseline modèle avec SHAP :**

**Tableau :**
- **Hypothétisé :** Il s'agit d'une colonne dont j'ai hypothétisé ou non comme étant responsable du churn (positivement ou négativement)
- **Fortes valeurs :** impact d'une valeur forte sur le churn
- **Faibles valeurs :** impact d'une valeur faible sur le churn
- ⬆️ : impact le churn de manière positif
- ⬇️ : impact le churn de manière négative


| # | Feature | Hypothétisé | Fortes valeurs | Faibles valeurs |
|---|---------|:-----------:|:--------------:|:---------------:|
| 1 | nb_integrations | | ⬇️⬇️⬇️ | ⬆️⬆️⬆️ |
| 2 | anciennete_mois | | ⬇️⬇️⬇️ | ⬆️⬆️⬆️ |
| 3 | derniere_connexion_jours | | ⬆️⬆️⬆️ | ⬇️⬇️ |
| 4 | tickets_support_90j | ✅ | ⬆️⬆️⬆️ | ⬇️⬇️⬇️ |
| 5 | csat | ✅ | ⬇️⬇️ | ⬆️⬆️ |
| 6 | retards_paiement_12m | ✅ | ⬆️⬆️ | ⬇️⬇️ |
| 7 | taux_adoption_pct | | ⬇️⬇️ | ⬆️⬆️ |
| 8 | delai_reponse_support_h | ✅ | ⬆️⬆️ | ⬇️⬇️ |
| 9 | plan_str | | ⬆️⬆️ | ⬇️ |
| 10 | revenu_mensuel_recurrent_eur | | ⬆️⬆️ | ⬇️ |
| 11 | utilisateurs_actifs | ✅ | ⬇️⬇️ | ⬆️ |
| 12 | fonctionnalites_utilisees | ✅ | ⬇️⬇️ | ⬆️ |
| 13 | plan_ent | | ⬇️⬇️⬇️ | ⬆️ |
| 14 | couleur_theme_interface_v | | ⬇️⬇️ | ⬆️ |
| 15 | taille_entreprise_pme | ✅ | ⬇️ | ⬆️ |
| 16 | heures_usage_30j | | ⬇️ | ⬆️ |
| 17 | niveau_anciennete_ancien | | ⬆️ | ⬇️ |
| 18 | secteur_fi | | ⬇️⬇️ | ⬆️ |
| 19 | taux_fonctionnalites | | ⬆️ | ⬇️ |
| 20 | niveau_anciennete_recent | | ⬇️⬇️ | ⬆️ |

### 10.2. Classification — recherche des hyperparamètres optimaux (XGBoost)

Même cible, même découpage et mêmes exclusions que la baseline : les deux
modèles restent comparables.

La cellule enchaîne deux étapes : **recherche** sur les neuf hyperparamètres
suivants plus le seuil de décision, puis **réentraînement** avec les valeurs
retenues et mesure sur le jeu de test tenu à l'écart jusque-là — c'est ce
modèle qui est versionné dans `artifacts/` — puis les deux lectures **SHAP**
des contributions, comme pour la baseline. Les valeurs viennent ici de
`TreeExplainer`, qui parcourt les arbres au lieu de supposer un effet
linéaire.

| Hyperparamètre | Plage explorée | Rôle |
|---|---|---|
| `n_estimators` | 100 → 800 | Nombre d'arbres |
| `max_depth` | 2 → 8 | Profondeur maximale d'un arbre |
| `learning_rate` | 0.01 → 0.3 | Pas d'apprentissage entre deux arbres |
| `subsample` | 0.6 → 1.0 | Part des lignes tirées par arbre |
| `colsample_bytree` | 0.4 → 1.0 | Part des colonnes tirées par arbre |
| `min_child_weight` | 1 → 20 | Poids minimal d'une feuille |
| `gamma` | 0.0 → 5.0 | Gain minimal exigé pour un découpage |
| `reg_alpha` | 1e-8 → 10 | Pénalité L1 sur les poids des feuilles |
| `reg_lambda` | 1e-8 → 10 | Pénalité L2 sur les poids des feuilles |
| `seuil` | 0.1 → 0.9 | Probabilité à partir de laquelle on alerte |

Les Hyperparamètres finaux et le seuil sont les suivants :


| Hyperparamètre | Valeur |
|---|---:|
| `seuil` | 0.40 |
| `n_estimators` | 300 |
| `max_depth` | 2 |
| `learning_rate` | 0.02 |
| `subsample` | 0.96 |
| `colsample_bytree` | 0.79 |
| `min_child_weight` | 8 |
| `gamma` | 4.58 |
| `reg_alpha` | 0.00 |
| `reg_lambda` | 3.58 |

Performances observées, sur la validation qui a servi à retenir ces valeurs
puis sur le test tenu à l'écart :

| Métrique | Validation | Test |
|---|---:|---:|
| score F2 | 0.78 | 0.78 |
| recall | 0.87 | 0.88 |
| false_positive_rate | 0.29 | 0.29 |
| precision | 0.54 | 0.54 |
| accuracy | 0.76 | 0.75 |
| auc | 0.88 | 0.87 |

Conclusion : l'écart entre validation et test est nul à deux décimales, la
recherche ne s'est donc pas sur-ajustée à la validation.


In [ ]:
from ml_churn.training.classification.final.classification_xgboost_training import (
    train_classification_xgboost,
)
from ml_churn.training.classification.final.classification_xgboost_tuning import (
    tune_xgboost,
)
from ml_churn.training.common.explain import (
    display_figure,
    shap_bar_figure,
    shap_summary_figure,
)

# 1. Recherche des hyperparametres : un modele reentraine par essai.
resultat_tuning_xgboost = tune_xgboost(n_essais=100)

# 2. Reentrainement avec les hyperparametres retenus, puis mesure sur le jeu de
# test tenu a l'ecart. Ce modele n'est pas enregistre : seul le modele reduit
# de la section suivante l'est.
resultat_xgboost = train_classification_xgboost(
    seuil=resultat_tuning_xgboost.seuil,
    hyperparametres=resultat_tuning_xgboost.hyperparametres,
    enregistrer=False,
)

# 3. Explication du modele sur le jeu de test.
display_figure(
    shap_bar_figure(
        resultat_xgboost.model,
        resultat_xgboost.X_test,
        titre="SHAP XGBoost — importance globale",
        max_features=None,  # toutes les colonnes, sans regroupement
    )
)
display_figure(
    shap_summary_figure(
        resultat_xgboost.model,
        resultat_xgboost.X_test,
        titre="SHAP XGBoost — effet par client",
    )
)

### 10.3. Classification — XGBoost, modèle réduit

De nombreuses features ont un très faible impact sur le modèle établie précédement, on va retirer les features suivantes :

| Colonne retirée | Modalités concernées |
|---|---|
| `groupe_experimentation` | `_a`, `_b`, `_c` |
| `code_datacenter` | `_eu_w1`, `_eu_w3`, `_us_e1`, `_ap_s1` |
| `couleur_theme_interface` | `_c`, `_ve`, `_b`, `_v`, `_s` |
| `plan` | `_str`, `_pro`, `_bus`, `_ent` |
| `taille_entreprise` | `_tpe`, `_pme`, `_eti`, `_ge` |
| `pays` | `_fr`, `_es`, `_ca`, `_de`, `_ch`, `_be` |
| `jour_souscription` | `_l`, `_m`, `_me`, `_j`, `_v`, `_s`, `_d` |
| `secteur` | `_te`, `_fi`, `_co`, `_sa`, `_in`, `_pb`, `_en` |

Entrainement d'un nouveau modèle avec exclusion des features :

Performances observées sur le test set, face au modèle baseline :

| Métrique | Baseline (seuil 0.5) | XGBoost réduit (seuil 0.4) |
|---|---:|---:|
| features | 64 | 24 |
| accuracy | 0.81 | 0.76 |
| recall | 0.80 | 0.87 |
| false_positive_rate | 0.19 | 0.29 |
| precision | 0.62 | 0.54 |
| auc | 0.88 | 0.87 |

Conclusion : le modèle XGBoost optimisé arrive à avoir une capacité de détection des churn légérement meilleur que le baseline (recall +7%) mais au prix de plus d'alertes (+10%).
On constate également que les features polarité_X (correspondant aux commentaires csm) ont aucun d'impact sur le modèle ce qui est contre-intuitif étant donné la pertinance évidante de cette donnée.


In [ ]:
from ml_churn.training.classification.final.classification_xgboost_training import (
    COLONNES_RETIREES,
    exclusions_sans,
    train_classification_xgboost,
)
from ml_churn.training.common.explain import (
    display_figure,
    shap_bar_figure,
    shap_summary_figure,
)

resultat_xgboost_reduit = train_classification_xgboost(
    seuil=resultat_tuning_xgboost.seuil,
    hyperparametres=resultat_tuning_xgboost.hyperparametres,
    exclusions=exclusions_sans(COLONNES_RETIREES),
)

# Explication du modele reduit sur le jeu de test.
display_figure(
    shap_bar_figure(
        resultat_xgboost_reduit.model,
        resultat_xgboost_reduit.X_test,
        titre="SHAP XGBoost réduit — importance globale",
        max_features=None,  # toutes les colonnes, sans regroupement
    )
)
display_figure(
    shap_summary_figure(
        resultat_xgboost_reduit.model,
        resultat_xgboost_reduit.X_test,
        titre="SHAP XGBoost réduit — effet par client",
    )
)